In [ ]:
import torch
import torch.nn as nn
import numpy as np

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Convolutional layer:
        # in_channels=1 for grayscale input, out_channels=16 (you can change this), kernel_size=3,
        # stride=1, and padding=1 to keep the spatial dimensions the same (32x32)
        self.conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=1)
        
        # Max pooling layer: kernel_size=2 and stride=2 reduce the spatial dimensions by half.
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Leaky ReLU activation:
        self.leaky_relu = nn.LeakyReLU(negative_slope=0.01)

    def forward(self, x):
        # Pass input through convolutional layer
        x = self.conv(x)
        # Then apply max pooling
        x = self.pool(x)
        # Finally apply Leaky ReLU activation
        x = self.leaky_relu(x)
        return x

# Create an instance of the model
model = SimpleCNN()

# Access the convolutional layer weights
# model.conv.weight has the shape [out_channels, in_channels, kernel_height, kernel_width]
weights = model.conv.weight

# Convert the weights to a NumPy array
weights_np = weights.detach().cpu().numpy()

# Optionally, set NumPy print options to see the full array
np.set_printoptions(threshold=np.inf)

#这里打印kernel weight的数据
print("Convolutional layer weights as array:")
print(weights_np)




Convolutional layer weights as array:
[[[[ 0.10652475  0.05705401 -0.0311929 ]
   [-0.00331509 -0.09943819 -0.20402412]
   [ 0.3243959   0.08743918 -0.14249381]]]]
Convolutional layer weights as array(fixed number):
[[[[   0.107    0.057   -0.031]
   [  -0.003   -0.099   -0.204]
   [   0.324    0.087   -0.142]]]]


打印fixed number

In [16]:
# ------------------------------------------------------------------
# Quantization parameters for fixed-point representation:
# Total bits = 8, Fractional bits = 3
total_bits = 8
frac_bits = 3
scale = 2 ** frac_bits

# For an 8-bit signed number, the range is:
min_val = -(1 << (total_bits - 1))       # -128
max_val = (1 << (total_bits - 1)) - 1      # 127

# Quantize each weight: multiply by scale, round, and clip to range.
quantized_int = np.clip(np.round(weights_np * scale), min_val, max_val).astype(np.int8)

# Recover the fixed-point float representation:
quantized_float = quantized_int / scale

# Print the fixed-point float representation using the specified format.
total_width = 8
decimal_places = 3
formatter_float = {'float_kind': lambda x: f"{x:{total_width}.{decimal_places}f}"}
formatted_fixed = np.array2string(quantized_float, formatter=formatter_float)
print("Quantized fixed-point weights (float representation):")
print(formatted_fixed)

# ------------------------------------------------------------------
# Functions to convert an 8-bit signed integer to binary and hexadecimal strings,
# ensuring that each printed string has the same fixed length.

def int8_to_bin(x):
    # Convert a signed int8 to its two's complement unsigned representation.
    x_unsigned = x if x >= 0 else (1 << total_bits) + x
    return format(x_unsigned, f'0{total_bits}b')

def int8_to_hex(x):
    # Convert a signed int8 to its two's complement unsigned representation.
    x_unsigned = x if x >= 0 else (1 << total_bits) + x
    # For 8 bits, we want exactly 2 hex digits.
    return format(x_unsigned, '02x')

# Vectorize the conversion functions so they can be applied elementwise.
vec_int8_to_bin = np.vectorize(int8_to_bin)
vec_int8_to_hex = np.vectorize(int8_to_hex)

# Create arrays of binary and hexadecimal strings for the quantized weights.
bin_array = vec_int8_to_bin(quantized_int)
hex_array = vec_int8_to_hex(quantized_int)

print("\nQuantized weights in binary (8 bits):")
print(np.array2string(bin_array, separator=', '))

print("\nQuantized weights in hexadecimal (2 hex digits):")
print(np.array2string(hex_array, separator=', '))

Quantized fixed-point weights (float representation):
[[[[   0.125    0.000    0.000]
   [   0.000   -0.125   -0.250]
   [   0.375    0.125   -0.125]]]]

Quantized weights in binary (8 bits):
[[[['00000001', '00000000', '00000000'],
   ['00000000', '11111111', '11111110'],
   ['00000011', '00000001', '11111111']]]]

Quantized weights in hexadecimal (2 hex digits):
[[[['01', '00', '00'],
   ['00', 'ff', 'fe'],
   ['03', '01', 'ff']]]]
